In [1]:
# Cell 1: Install Dependencies
!pip install langgraph langchain langchain-openai pandas tabulate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 6.9 MB/s eta 0:00:00


In [2]:
# Cell 2: API Key Setup
import getpass
import os

os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API Key: ")
print("✅ API Key set.")

Enter your OpenRouter API Key: ··········
✅ API Key set.


In [4]:
# Cell 3: LLM Client (OpenRouter → z-ai/glm-4.5-air:free)  ← FIXED MODEL ID
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="z-ai/glm-4.5-air:free",        # ← was "z-ai/glm-4.5-free:air" (wrong)
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0
)

# Quick sanity check
response = llm.invoke("Say: LLM connected successfully.")
print(response.content)

LLM connected successfully.


In [5]:
# Cell 4: Generate Dummy Contacts CSV
import pandas as pd

data = {
    "name":         ["Alice Johnson", "Bob Smith", "Carol White", "David Brown",
                     "Eva Green", "Frank Lee", "Grace Kim", "Henry Davis",
                     "Iris Chen", "Jack Wilson"],
    "email":        ["alice@gmail.com", "bob@yahoo.com", "carol@gmail.com",
                     "david@company.com", "eva@gmail.com", "frank@outlook.com",
                     "grace@techcorp.com", "henry@yahoo.com", "iris@gmail.com",
                     "jack@startup.com"],
    "company_name": ["Acme Corp", "BrightTech", "Acme Corp", "DataWorks",
                     "BrightTech", "Startup Hub", "TechCorp", "DataWorks",
                     "Acme Corp", "Startup Hub"],
    "phone_number": ["9876543210", "9123456780", "9988776655", "9001122334",
                     "9871234567", "9765432100", "9654321009", "9543210098",
                     "9432100987", "9321009876"],
    "city":         ["Mumbai", "Delhi", "Mumbai", "Bangalore",
                     "Delhi", "Mumbai", "Bangalore", "Chennai",
                     "Mumbai", "Delhi"],
    "designation":  ["Manager", "Engineer", "Director", "Analyst",
                     "Engineer", "Founder", "Engineer", "Manager",
                     "Analyst", "Founder"]
}

df = pd.DataFrame(data)
df.to_csv("contacts.csv", index=False)
print("✅ contacts.csv created!")
print(df.head())

✅ contacts.csv created!
            name              email company_name phone_number       city  \
0  Alice Johnson    alice@gmail.com    Acme Corp   9876543210     Mumbai   
1      Bob Smith      bob@yahoo.com   BrightTech   9123456780      Delhi   
2    Carol White    carol@gmail.com    Acme Corp   9988776655     Mumbai   
3    David Brown  david@company.com    DataWorks   9001122334  Bangalore   
4      Eva Green      eva@gmail.com   BrightTech   9871234567      Delhi   

  designation  
0     Manager  
1    Engineer  
2    Director  
3     Analyst  
4    Engineer  


In [6]:
# Cell 5: State Definition
from typing import TypedDict, Annotated, List, Any
import operator

class QueryGraphState(TypedDict):
    # Input
    user_query: str
    conversation_history: Annotated[List[dict], operator.add]  # reducer: always append

    # Understanding
    query_intent: str          # filter | count | lookup | aggregate | unknown

    # Execution
    generated_pandas_code: str
    query_result: Any
    query_error: str
    retry_count: int

    # Output
    final_response: str

    # Meta (loaded once)
    dataframe_schema: str

print("✅ State defined.")

✅ State defined.


In [7]:
# Cell 6: Tools

def get_dataframe_schema(df: pd.DataFrame) -> str:
    """Returns column names, dtypes, and 3 sample rows as a string for LLM context."""
    schema = f"Columns: {list(df.columns)}\n"
    schema += f"Dtypes:\n{df.dtypes.to_string()}\n\n"
    schema += f"Sample rows:\n{df.head(3).to_string(index=False)}"
    return schema


def execute_pandas_query(code: str, df: pd.DataFrame) -> dict:
    """Safely executes a pandas code string. Returns result or error."""
    try:
        local_vars = {"df": df, "pd": pd}
        result = eval(code.strip(), {"__builtins__": {}}, local_vars)
        return {"success": True, "result": result}
    except Exception as e:
        return {"success": False, "error": str(e)}


def format_result(result: Any) -> str:
    """Converts dataframe/series/scalar to a readable string."""
    if isinstance(result, pd.DataFrame):
        if result.empty:
            return "No matching records found."
        return result.to_markdown(index=False)
    elif isinstance(result, pd.Series):
        return result.to_string()
    elif result is None:
        return "No result."
    else:
        return str(result)


print("✅ Tools defined.")

✅ Tools defined.


In [8]:
# Cell 7: All Nodes

# ── Node 1: Schema Loader ──────────────────────────────────────────────────
def schema_loader_node(state: QueryGraphState) -> dict:
    """Loads CSV and injects schema into state. Runs once at graph start."""
    df = pd.read_csv("contacts.csv")
    schema = get_dataframe_schema(df)
    print("📋 [SchemaLoader] Schema loaded.")
    return {"dataframe_schema": schema}


# ── Node 2: Intent Classifier ──────────────────────────────────────────────
def intent_classifier_node(state: QueryGraphState) -> dict:
    """LLM classifies what the user wants to do with the data."""
    history_text = ""
    recent_history = state.get("conversation_history", [])[-10:]  # last 10 turns
    if recent_history:
        history_text = "\n".join(
            [f"{m['role'].upper()}: {m['content']}" for m in recent_history]
        )

    prompt = f"""You are analyzing a query over a CSV contacts dataset.

Schema:
{state['dataframe_schema']}

Conversation so far:
{history_text if history_text else "None"}

Current user query: "{state['user_query']}"

Classify the intent as exactly ONE of these words:
- filter     → user wants rows matching a condition
- count      → user wants a number or count
- lookup     → user wants a specific field value for a known record
- aggregate  → user wants sum/avg/max/min/group-by
- unknown    → query cannot be answered from this data

Return ONLY the single intent word, nothing else."""

    response = llm.invoke(prompt)
    intent = response.content.strip().lower().split()[0]

    valid_intents = {"filter", "count", "lookup", "aggregate", "unknown"}
    if intent not in valid_intents:
        intent = "unknown"

    print(f"🎯 [IntentClassifier] Intent: {intent}")
    return {"query_intent": intent}


# ── Node 3: Pandas Code Generator ─────────────────────────────────────────
def pandas_code_generator_node(state: QueryGraphState) -> dict:
    """LLM generates a pandas one-liner based on user query and schema."""
    recent_history = state.get("conversation_history", [])[-10:]
    history_text = "\n".join(
        [f"{m['role'].upper()}: {m['content']}" for m in recent_history]
    ) if recent_history else "None"

    prompt = f"""You are a pandas expert. Generate pandas code to answer the user's query.

Dataframe variable name: df
Schema:
{state['dataframe_schema']}

Conversation history (for context on follow-up queries):
{history_text}

Current query: "{state['user_query']}"
Intent: {state['query_intent']}

Rules:
1. Return ONLY a single line of executable pandas code.
2. Do NOT include print(), display(), or markdown — just the expression.
3. String comparisons must be case-insensitive using .str.lower().
4. For follow-up queries like "sort those" or "filter those", refer to prior context.

Example outputs:
df[df['city'].str.lower() == 'mumbai']
df['company_name'].value_counts()
df[df['email'].str.contains('gmail', case=False)].shape[0]"""

    response = llm.invoke(prompt)
    code = response.content.strip().strip("```").strip("python").strip()
    print(f"💻 [CodeGenerator] Code: {code}")
    return {"generated_pandas_code": code, "query_error": None, "retry_count": state.get("retry_count", 0)}


# ── Node 4: Query Executor ─────────────────────────────────────────────────
def query_executor_node(state: QueryGraphState) -> dict:
    """Executes the generated pandas code against the actual dataframe."""
    df = pd.read_csv("contacts.csv")
    result = execute_pandas_query(state["generated_pandas_code"], df)

    if result["success"]:
        print(f"✅ [QueryExecutor] Query succeeded.")
        return {"query_result": result["result"], "query_error": None}
    else:
        retry = state.get("retry_count", 0) + 1
        print(f"❌ [QueryExecutor] Error (attempt {retry}): {result['error']}")
        return {"query_result": None, "query_error": result["error"], "retry_count": retry}


# ── Node 5: Error Recovery ─────────────────────────────────────────────────
def error_recovery_node(state: QueryGraphState) -> dict:
    """LLM looks at the error and fixes the pandas code."""
    prompt = f"""The following pandas code failed with an error. Fix it.

Schema:
{state['dataframe_schema']}

Failed code:
{state['generated_pandas_code']}

Error message:
{state['query_error']}

Original query: "{state['user_query']}"

Return ONLY the corrected single-line pandas code, nothing else."""

    response = llm.invoke(prompt)
    fixed_code = response.content.strip().strip("```").strip("python").strip()
    print(f"🔧 [ErrorRecovery] Fixed code: {fixed_code}")
    return {"generated_pandas_code": fixed_code, "query_error": None}


# ── Node 6: Response Formatter ─────────────────────────────────────────────
def response_formatter_node(state: QueryGraphState) -> dict:
    """Formats the raw result into a conversational human-readable response."""

    # Handle unknown intent
    if state.get("query_intent") == "unknown":
        response = "I can only answer questions about the contacts data (names, emails, companies, phones, cities, designations). Could you rephrase your question?"
        print(f"💬 [ResponseFormatter] Unknown intent response.")
        return {
            "final_response": response,
            "conversation_history": [
                {"role": "user", "content": state["user_query"]},
                {"role": "assistant", "content": response}
            ]
        }

    # Handle exhausted retries
    if state.get("query_error") and state.get("retry_count", 0) >= 3:
        response = f"I wasn't able to process that query after multiple attempts. Try rephrasing it or simplifying the request."
        print(f"💬 [ResponseFormatter] Retry exhausted response.")
        return {
            "final_response": response,
            "conversation_history": [
                {"role": "user", "content": state["user_query"]},
                {"role": "assistant", "content": response}
            ]
        }

    formatted = format_result(state.get("query_result"))

    prompt = f"""You are a helpful data assistant. The user asked a question about a contacts CSV.

User question: "{state['user_query']}"

Query result:
{formatted}

Write a clear, friendly, conversational answer based on this result.
If it's a table, present it as-is. If it's a number, explain it naturally.
Keep it concise."""

    response = llm.invoke(prompt)
    final = response.content.strip()
    print(f"💬 [ResponseFormatter] Response ready.")

    return {
        "final_response": final,
        "conversation_history": [
            {"role": "user", "content": state["user_query"]},
            {"role": "assistant", "content": final}
        ]
    }


print("✅ All nodes defined.")

✅ All nodes defined.


In [9]:
# Cell 8: Conditional Edge Functions

def route_after_intent(state: QueryGraphState) -> str:
    """After intent classification: skip to formatter if unknown, else generate code."""
    if state.get("query_intent") == "unknown":
        print("🔀 [Edge] Intent=unknown → response_formatter")
        return "response_formatter_node"
    print("🔀 [Edge] Intent valid → pandas_code_generator")
    return "pandas_code_generator_node"


def route_after_execution(state: QueryGraphState) -> str:
    """After query execution: go to formatter on success, else check retry."""
    if state.get("query_error") is None:
        print("🔀 [Edge] Execution success → response_formatter")
        return "response_formatter_node"
    print("🔀 [Edge] Execution failed → retry_gate")
    return "retry_gate"


def retry_gate(state: QueryGraphState) -> str:
    """Decide whether to retry error recovery or give up."""
    if state.get("retry_count", 0) < 3:
        print(f"🔀 [Edge] Retrying... attempt {state['retry_count']}")
        return "error_recovery_node"
    print("🔀 [Edge] Max retries reached → response_formatter")
    return "response_formatter_node"


print("✅ Edge functions defined.")

✅ Edge functions defined.


In [11]:
# Cell 9: Build the LangGraph  ← FIXED
from langgraph.graph import StateGraph, END

builder = StateGraph(QueryGraphState)

# ── Add all nodes ──────────────────────────────────────────────────────────
builder.add_node("schema_loader_node",         schema_loader_node)
builder.add_node("intent_classifier_node",     intent_classifier_node)
builder.add_node("pandas_code_generator_node", pandas_code_generator_node)
builder.add_node("query_executor_node",        query_executor_node)
builder.add_node("error_recovery_node",        error_recovery_node)
builder.add_node("response_formatter_node",    response_formatter_node)

# ── Entry point ────────────────────────────────────────────────────────────
builder.set_entry_point("schema_loader_node")

# ── Fixed edges ────────────────────────────────────────────────────────────
builder.add_edge("schema_loader_node",          "intent_classifier_node")
builder.add_edge("pandas_code_generator_node",  "query_executor_node")
builder.add_edge("error_recovery_node",         "query_executor_node")
builder.add_edge("response_formatter_node",     END)

# ── Conditional edge: after intent classification ──────────────────────────
builder.add_conditional_edges(
    "intent_classifier_node",
    route_after_intent,
    {
        "pandas_code_generator_node": "pandas_code_generator_node",
        "response_formatter_node":    "response_formatter_node"
    }
)

# ── Conditional edge: after execution (success vs fail+retry in one function) ──
def route_after_execution(state: QueryGraphState) -> str:
    if state.get("query_error") is None:
        print("🔀 [Edge] Execution success → response_formatter")
        return "response_formatter_node"
    if state.get("retry_count", 0) < 3:
        print(f"🔀 [Edge] Execution failed, retrying (attempt {state['retry_count']})")
        return "error_recovery_node"
    print("🔀 [Edge] Max retries reached → response_formatter")
    return "response_formatter_node"

builder.add_conditional_edges(
    "query_executor_node",
    route_after_execution,
    {
        "response_formatter_node": "response_formatter_node",
        "error_recovery_node":     "error_recovery_node"
    }
)

# ── Compile ────────────────────────────────────────────────────────────────
graph = builder.compile()
print("✅ Graph compiled successfully!")

✅ Graph compiled successfully!


In [ ]:
# Cell 11: Single Query Test
initial_state = {
    "user_query": "Show companies having owner like designation",
    "conversation_history": [],
    "query_intent": "",
    "generated_pandas_code": "",
    "query_result": None,
    "query_error": None,
    "retry_count": 0,
    "final_response": "",
    "dataframe_schema": ""
}

result = graph.invoke(initial_state)

print("\n" + "="*50)
print("🤖 Assistant:", result["final_response"])
print("="*50)

📋 [SchemaLoader] Schema loaded.
🎯 [IntentClassifier] Intent: filter
🔀 [Edge] Intent valid → pandas_code_generator
